In [ ]:
#!pip install sentence_transformers
#!pip install ag2[retrievechat]
#!pip install pyautogen
#!pip install chromadb
#https://microsoft.github.io/autogen/0.2/docs/notebooks/agentchat_RetrieveChat/

# Setting base for Retrive Chat

In [1]:
import autogen

print(autogen.__version__)
print(autogen.__file__)

0.13.4
/Users/kavin/Agentic-ai-practice/AI-venv/lib/python3.13/site-packages/autogen/__init__.py


In [ ]:
import json
import os

import chromadb

import autogen
from autogen import AssistantAgent
from autogen.agentchat.contrib.retrieve_user_proxy_agent import RetrieveUserProxyAgent

# Accepted file formats for that can be stored in
# a vector database instance
from autogen.retrieve_utils import TEXT_FORMATS

config_list = [
    {
        "model": "gpt-4o-mini",
        "api_key": "openai"
    }
]

assert len(config_list) > 0
print("models to use: ", [config_list[i]["model"] for i in range(len(config_list))])

models to use:  ['gpt-4o-mini']


In [3]:
print("Accepted file formats for `docs_path`:")
print(TEXT_FORMATS)

Accepted file formats for `docs_path`:
['txt', 'json', 'csv', 'tsv', 'md', 'html', 'htm', 'rtf', 'rst', 'jsonl', 'log', 'xml', 'yaml', 'yml', 'pdf', 'mdx']


In [4]:
config_list = [
    {
        "api_type": "openai",
        "model": "gpt-4o-mini",
        "api_key": "OPENAPI"
    }
]

In [5]:
import agentops
agentops.init("API-Agentops")

🖇 AgentOps: [ConcurrentFuturesInstrumentor] Successfully instrumented concurrent.futures.ThreadPoolExecutor
🖇 AgentOps: You're on the agentops free plan 🤔


In [6]:
# 1. create an AssistantAgent instance named "assistant"
assistant = AssistantAgent(
    name="assistant",
    system_message="You are a helpful assistant.",
    llm_config={
        "timeout": 600,
        "cache_seed": 42,
        "config_list": config_list,
    },
)


In [7]:
from autogen.agentchat.contrib.retrieve_user_proxy_agent import RetrieveUserProxyAgent

# Extract text from local PDF files
pdf_files = ["book.pdf"]
#docs_content = [extract_text_from_pdf(file) for file in pdf_files]

from autogen.retrieve_utils import TEXT_FORMATS,extract_text_from_pdf
docs_content= extract_text_from_pdf("book.pdf")

In [8]:
# Define the RetrieveUserProxyAgent instance
ragproxyagent = RetrieveUserProxyAgent(
    name="ragproxyagent",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=3,
    retrieve_config={
        "task": "qa",
        "docs_path": ["autogen-docs/book.pdf"],
        "docs_content": docs_content,  # Provide the extracted content directly
        "chunk_token_size": 200,
        "model": config_list[0]["model"],
        "vector_db": "chroma",
        "overwrite": True,
        "get_or_create": True,
    },
    code_execution_config=False,
)

/Users/kavin/Agentic-ai-practice/AI-venv/lib/python3.13/site-packages/agentops/instrumentation/__init__.py:384: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  return _original_builtins_import(name, globals_dict, locals_dict, fromlist, level)
Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 9645.09it/s]


# Process Starts for Scenario 1 : Generate code based off docstrings w/o human feedback

In [9]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

qa_problem = "summarize the content and use python code to any calculation given in the book"
chat_result = ragproxyagent.initiate_chat(assistant, message=ragproxyagent.message_generator, problem=qa_problem)
agentops.end_session("Success")

Trying to create collection.


max_tokens is too small to fit a single line of text. Breaking this line:
	The 22 ...
Failed to split docs with must_break_at_empty_line being True, set to False.
2026-06-18 23:33:16,947 - autogen.agentchat.contrib.retrieve_user_proxy_agent - INFO - Found 215 chunks.
2026-06-18 23:33:16,953 - autogen.agentchat.contrib.vectordb.chromadb - INFO - No content embedding is provided. Will use the VectorDB's embedding function to generate the content embedding.


VectorDB returns doc_ids:  [['fe1105cf', '8bf7f8ee', '1d0894aa', '839a2e04', 'd11d708f', '150c596a', 'fbda2b4c', '59e93e46', '1613971f', '25edaaee', '7091167f', 'c1c95a95', '43a28a91', '16a1e2eb', 'e7d73411', 'a5e9b671', 'a1061522', '05dac356', '1097b211', 'd41e75dd']]
Adding content of doc fe1105cf to context.
Adding content of doc 8bf7f8ee to context.
Adding content of doc 1d0894aa to context.
Adding content of doc 839a2e04 to context.
Adding content of doc d11d708f to context.
Adding content of doc 150c596a to context.
Adding content of doc fbda2b4c to context.
Adding content of doc 59e93e46 to context.
Adding content of doc 1613971f to context.
Adding content of doc 25edaaee to context.
Adding content of doc 7091167f to context.
Adding content of doc c1c95a95 to context.
Adding content of doc 43a28a91 to context.
Adding content of doc 16a1e2eb to context.
Adding content of doc e7d73411 to context.
Adding content of doc a5e9b671 to context.
Adding content of doc a1061522 to context.

assistant (to ragproxyagent):

The context discusses the principles of marketing, with a particular focus on how perceptions influence consumer behavior and the market dynamics. It highlights the importance of being first in a category, the pitfalls of line extensions, and the distinction between product-driven marketing versus perception-driven marketing.

For calculations mentioned in the text, consider the sales of Miller High Life, which almost tripled from 8.6 million to 23.6 million barrels from 1974 to 1979. The growth can be calculated as follows:

```python
initial_sales = 8.6  # in million barrels
final_sales = 23.6  # in million barrels
growth = final_sales - initial_sales
percentage_increase = (growth / initial_sales) * 100

print("Sales Growth (Million Barrels):", growth)
print("Percentage Increase:", percentage_increase)
```

This code will provide the sales growth in million barrels and the percentage increase.

-----------------------------------------------------------

assistant (to ragproxyagent):

UPDATE CONTEXT

--------------------------------------------------------------------------------
Updating context and resetting conversation.
VectorDB returns doc_ids:  [['fe1105cf', '8bf7f8ee', '1d0894aa', '839a2e04', 'd11d708f', '150c596a', 'fbda2b4c', '59e93e46', '1613971f', '25edaaee', '7091167f', 'c1c95a95', '43a28a91', '16a1e2eb', 'e7d73411', 'a5e9b671', 'a1061522', '05dac356', '1097b211', 'd41e75dd', '320d8488', '52a863ba', '10e789b5', 'cd89bf11', 'ef80c94d', 'd0e14ddf', '3f2e644d', 'fc6e8864', '2fe1e962', '62e6169a', '0a1d7e6c', '5f2c6c84', 'c1b21496', '0e403d83', 'e75197c0', '54fb1eca', 'a683764c', 'cf32340a', '546ddb88', '72117217', '9e2f60a8', 'a7c23eee', 'a4663f36', 'f5a5ba4c', '9a5cca9b', '80d28530', '217a91d7', 'f025a835', '1dcef28c', '2e0213ff', 'be843ac3', 'd5a791b1', '9d9cdd69', '5b19ade7', '9cdc0320', '4359e2a4', 'a99c802c', 'b6e7f27a', '82eebd0a', 'a3efc671']]
Adding content of doc 320d8488 to context.
Adding content of doc 52a863ba to 

assistant (to ragproxyagent):

UPDATE CONTEXT

--------------------------------------------------------------------------------
Updating context and resetting conversation.
Adding content of doc 9e2f60a8 to context.
Adding content of doc a7c23eee to context.
Adding content of doc a4663f36 to context.
Adding content of doc f5a5ba4c to context.
Adding content of doc 9a5cca9b to context.
Adding content of doc 80d28530 to context.
Adding content of doc 217a91d7 to context.
Adding content of doc f025a835 to context.
Adding content of doc 1dcef28c to context.
Adding content of doc 2e0213ff to context.
Adding content of doc be843ac3 to context.
Adding content of doc d5a791b1 to context.
Adding content of doc 9d9cdd69 to context.
Adding content of doc 5b19ade7 to context.
Adding content of doc 9cdc0320 to context.
Adding content of doc 4359e2a4 to context.
Adding content of doc a99c802c to context.
Adding content of doc b6e7f27a to context.
Adding content of doc 82eebd0a to context.
Adding con

assistant (to ragproxyagent):

The context discusses several marketing principles outlined in the book "22 Laws of Marketing." Key content includes:

1. **Law of Sacrifice** - To succeed, companies must give up certain product lines or target markets rather than expanding.
2. **Law of the Mind** - Perceptions matter more than reality; being first can lead to success.
3. **Law of Line Extension** - Expanding a brand's product line often leads to brand dilution and failure, as exemplified by 7-Up's reduced market share despite flavor extensions.
4. **Examples of Companies** - Discusses brands like IBM and Coca-Cola, illustrating how overextension can lead to failure.

No specific calculations are presented in the content. Hence, no Python code for calculations is provided.

For further clarification or specific information, please indicate.

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (b73c1f51-37f6-40c8-8068-dcaf6fa31cc0): M

🖇 AgentOps: end_session() is deprecated and will be removed in v4 in the future. Use agentops.end_trace() instead.
🖇 AgentOps: end_session called but no active trace context found.


In [ ]:
#agentops.end_session("Success")

# Process Starts for Scenario 2 : Answer a question based off docstrings w/o human feedback

In [10]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

qa_problem = "You are a creative head . do many brainstorms"
chat_result = ragproxyagent.initiate_chat(assistant, message=ragproxyagent.message_generator, problem=qa_problem)

VectorDB returns doc_ids:  [['1097b211', '7e1ac0cd', '0e08a976', '81d8712b', 'be843ac3', 'a5e9b671', 'b2f1dbea', 'ba9a27e8', '21ec236c', '8126c05a', 'cf32340a', '7e3514a1', '89fd1dac', '5b19ade7', '10e789b5', '0e403d83', '71a3bd57', '1e7668a9', 'a1f246ff', '4cf44936']]
Adding content of doc 1097b211 to context.
Adding content of doc 7e1ac0cd to context.
Adding content of doc 0e08a976 to context.
Adding content of doc 81d8712b to context.
Adding content of doc be843ac3 to context.
Adding content of doc a5e9b671 to context.
Adding content of doc b2f1dbea to context.
Adding content of doc ba9a27e8 to context.
Adding content of doc 21ec236c to context.
Adding content of doc 8126c05a to context.
Adding content of doc cf32340a to context.
Adding content of doc 7e3514a1 to context.
Adding content of doc 89fd1dac to context.
Adding content of doc 5b19ade7 to context.
Adding content of doc 10e789b5 to context.
Adding content of doc 0e403d83 to context.
Adding content of doc 71a3bd57 to context.

assistant (to ragproxyagent):

UPDATE CONTEXT

--------------------------------------------------------------------------------
Updating context and resetting conversation.
VectorDB returns doc_ids:  [['1097b211', '7e1ac0cd', '0e08a976', '81d8712b', 'be843ac3', 'a5e9b671', 'b2f1dbea', 'ba9a27e8', '21ec236c', '8126c05a', 'cf32340a', '7e3514a1', '89fd1dac', '5b19ade7', '10e789b5', '0e403d83', '71a3bd57', '1e7668a9', 'a1f246ff', '4cf44936', 'e47994d3', 'c1c95a95', 'c41df90f', '083871f5', '54fb1eca', 'e7c47c06', 'd5a791b1', 'e62f1169', '0a1d7e6c', 'd41e75dd', 'c062874b', '526bc106', '51f3c69e', 'ef80c94d', '7091167f', '788184ee', 'fc6e8864', 'f5a5ba4c', 'f5837698', '190d163f', 'ed7e9058', '039e8163', 'c1b21496', 'd11d708f', '2e0213ff', '9d9cdd69', '8bf7f8ee', 'd90d06bc', 'd6c8a6b5', 'cb09fdfc', '25edaaee', '9bb6d80c', '8f1eecad', '7c55e0a7', '217a91d7', '0c4927c1', '52a863ba', '696c1f83', 'f025a835', 'ed34a0f7']]
Adding content of doc e47994d3 to context.
Adding content of doc c1c95a95 to 

assistant (to ragproxyagent):

UPDATE CONTEXT

--------------------------------------------------------------------------------
Updating context and resetting conversation.
Adding content of doc ed7e9058 to context.
Adding content of doc 039e8163 to context.
Adding content of doc c1b21496 to context.
Adding content of doc d11d708f to context.
Adding content of doc 2e0213ff to context.
Adding content of doc 9d9cdd69 to context.
Adding content of doc 8bf7f8ee to context.
Adding content of doc d90d06bc to context.
Adding content of doc d6c8a6b5 to context.
Adding content of doc cb09fdfc to context.
Adding content of doc 25edaaee to context.
Adding content of doc 9bb6d80c to context.
Adding content of doc 8f1eecad to context.
Adding content of doc 7c55e0a7 to context.
Adding content of doc 217a91d7 to context.
Adding content of doc 0c4927c1 to context.
Adding content of doc 52a863ba to context.
Adding content of doc 696c1f83 to context.
Adding content of doc f025a835 to context.
Adding con

assistant (to ragproxyagent):

UPDATE CONTEXT

--------------------------------------------------------------------------------
Updating context and resetting conversation.
VectorDB returns doc_ids:  [['1097b211', '7e1ac0cd', '0e08a976', '81d8712b', 'be843ac3', 'a5e9b671', 'b2f1dbea', 'ba9a27e8', '21ec236c', '8126c05a', 'cf32340a', '7e3514a1', '89fd1dac', '5b19ade7', '10e789b5', '0e403d83', '71a3bd57', '1e7668a9', 'a1f246ff', '4cf44936', 'e47994d3', 'c1c95a95', 'c41df90f', '083871f5', '54fb1eca', 'e7c47c06', 'd5a791b1', 'e62f1169', '0a1d7e6c', 'd41e75dd', 'c062874b', '526bc106', '51f3c69e', 'ef80c94d', '7091167f', '788184ee', 'fc6e8864', 'f5a5ba4c', 'f5837698', '190d163f', 'ed7e9058', '039e8163', 'c1b21496', 'd11d708f', '2e0213ff', '9d9cdd69', '8bf7f8ee', 'd90d06bc', 'd6c8a6b5', 'cb09fdfc', '25edaaee', '9bb6d80c', '8f1eecad', '7c55e0a7', '217a91d7', '0c4927c1', '52a863ba', '696c1f83', 'f025a835', 'ed34a0f7']]
No more context, will terminate.
ragproxyagent (to assistant):

TERMINATE

--

# Process Starts for Scenario 3 : Generate code based off docstrings w/ human feedback

In [ ]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

# set `human_input_mode` to be `ALWAYS`, so the agent will ask for human input at every step.
ragproxyagent.human_input_mode = "ALWAYS"
code_problem = "Based on this book give Marketing idea for AI and Datascience trainig institute. Give 5 ideas. if any coding requires use python and save it as .py"
chat_result = ragproxyagent.initiate_chat(assistant, message=ragproxyagent.message_generator, problem=code_problem)

# Process Starts for Scenario 4: Answer a question based off docstrings w/ human feedback

In [ ]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

# set `human_input_mode` to be `ALWAYS`, so the agent will ask for human input at every step.
ragproxyagent.human_input_mode = "ALWAYS"
qa_problem = "Based on this book give Marketing idea for AI and Datascience trainig institute. Give 5 ideas. "
chat_result = ragproxyagent.initiate_chat(
    assistant, message=ragproxyagent.message_generator, problem=qa_problem
)  # type "exit" to exit the conversation